# Text-to-SQL experiment pipeline

Interactive notebook for the few-shot Text-to-SQL research slice in this repo.
The 34 Python modules stay importable; this notebook only orchestrates them.

**Flow:** schema graph / ICRL pool → single-query debug → batch eval → pandas tables → matplotlib charts → mismatch inspector.

Copy `.env.example` to `.env` and set `ROUTER_API_KEY` (or `OPENAI_API_KEY`) before live inference. Keys are never hardcoded here.

Set `LIMIT = 20` for a smoke run, or `LIMIT = None` for the full split. Set `RUN_INFERENCE = False` to plot saved `results/*.jsonl` / `mismatch_cases.json` without calling the router.


## 1. Setup & environment

Add the repo root to `sys.path`, enable autoreload, load `.env`, bootstrap a minimal in-memory Django settings object (this slice has no `aras_ai_data.settings`), then import pandas, matplotlib, and the research modules.


In [1]:
from __future__ import annotations

%load_ext autoreload
%autoreload 2

import json
import os
import sys
import types
from pathlib import Path
from types import SimpleNamespace

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "data_service").is_dir():
    REPO_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else REPO_ROOT
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")

import importlib.machinery
import importlib.util

import django
from django.apps import AppConfig
from django.conf import settings as dj_settings

if "data_service.apps" not in sys.modules:
    spec = importlib.machinery.ModuleSpec(
        "data_service.apps",
        loader=None,
        origin=str(REPO_ROOT / "data_service" / "apps.py"),
    )
    apps_mod = importlib.util.module_from_spec(spec)
    apps_mod.__file__ = spec.origin
    apps_mod.__package__ = "data_service"

    class DataServiceConfig(AppConfig):
        name = "data_service"
        label = "data_service"
        default = True

    apps_mod.DataServiceConfig = DataServiceConfig
    sys.modules["data_service.apps"] = apps_mod

if not dj_settings.configured:
    dj_settings.configure(
        SECRET_KEY="notebook-only-not-for-production",
        DEBUG=True,
        INSTALLED_APPS=["data_service.apps.DataServiceConfig"],
        DATABASES={
            "default": {
                "ENGINE": "django.db.backends.sqlite3",
                "NAME": ":memory:",
            }
        },
        DEFAULT_AUTO_FIELD="django.db.models.BigAutoField",
        ROUTER_BASE_URL=os.environ.get("ROUTER_BASE_URL", "http://127.0.0.1:8101"),
        ROUTER_API_KEY=os.environ.get("ROUTER_API_KEY")
        or os.environ.get("OPENAI_API_KEY")
        or "",
        SQL_ROW_LIMIT=int(os.environ.get("SQL_ROW_LIMIT", "100") or 100),
        SQL_TIMEOUT_SECONDS=int(os.environ.get("SQL_TIMEOUT_SECONDS", "30") or 30),
        FEWSHOT_RAG_TOP_K=int(os.environ.get("FEWSHOT_RAG_TOP_K", "5") or 5),
        USE_TZ=True,
    )
    django.setup()

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

import matplotlib.pyplot as plt
import pandas as pd

from benchmark.official_spider_metrics import score_sql, summarize_metrics
from data_service.agent.fewshot_skeleton import retrieve_for_prompt, sql_skeleton
from data_service.clients.router import RouterClient
from data_service.models import DataSource, DataSourceType, FewShotExample
from data_service.services import connections as connections_mod
from repro_config import load as load_repro_config
from repro_config import overlay as overlay_repro

try:
    from data_service.agent.icrl import build_traversals, retrieve_similar, schema_join_stats
    ICRL_IMPORT_ERROR = None
except Exception as exc:  # marshaled icrl.bin is version-sensitive
    ICRL_IMPORT_ERROR = exc
    sys.modules.pop("data_service.agent.icrl", None)
    retrieve_similar = retrieve_for_prompt

    def schema_join_stats(meta, *args, **kwargs):
        tables = (meta or {}).get("tables") or []
        pairs = 0
        for table in tables:
            if isinstance(table, dict):
                pairs += len(table.get("foreign_keys") or [])
        return len(tables), pairs

    def build_traversals(meta, max_traversals: int = 8, **kwargs):
        tables = (meta or {}).get("tables") or []
        out = []
        by_name = {}
        for table in tables:
            if not isinstance(table, dict):
                continue
            name = str(table.get("table") or "")
            cols = tuple(str(c) for c in (table.get("columns") or []))
            node = SimpleNamespace(
                name=name,
                tables=(name,),
                columns=cols,
                columns_by_table={name: cols},
                foreign_keys=(),
            )
            node.schema_snippet = (
                lambda n=name, c=cols: f"- {n}: {', '.join(c)}"
            )
            by_name[name] = node
            out.append(node)
        for table in tables:
            if not isinstance(table, dict):
                continue
            src = str(table.get("table") or "")
            for fk in table.get("foreign_keys") or []:
                dst = str(fk.get("references_table") or "")
                if src not in by_name or dst not in by_name:
                    continue
                left, right = by_name[src], by_name[dst]
                cols_map = {
                    src: left.columns,
                    dst: right.columns,
                }
                pair = SimpleNamespace(
                    name=f"{src}+{dst}",
                    tables=(src, dst),
                    columns=left.columns + right.columns,
                    columns_by_table=cols_map,
                    foreign_keys=(
                        (
                            src,
                            str(fk.get("column") or ""),
                            dst,
                            str(fk.get("references_column") or ""),
                        ),
                    ),
                )
                pair.schema_snippet = (
                    lambda s=src, d=dst, c=cols_map, e=pair.foreign_keys: (
                        "\n".join(f"- {n}: {', '.join(c[n])}" for n in (s, d))
                        + "\nJoins:\n"
                        + "\n".join(f"- {a}.{b} = {x}.{y}" for a, b, x, y in e)
                    )
                )
                out.append(pair)
                if len(out) >= int(max_traversals):
                    return out[: int(max_traversals)]
        return out[: int(max_traversals)]

    shim = types.ModuleType("data_service.agent.icrl")
    shim.build_traversals = build_traversals
    shim.retrieve_similar = retrieve_similar
    shim.schema_join_stats = schema_join_stats
    sys.modules["data_service.agent.icrl"] = shim
    print("icrl marshal import failed; using fewshot_skeleton.retrieve_for_prompt + tables.json graph fallback")
    print(f"  {type(exc).__name__}: {exc}")

from data_service.agent import pipeline

print("pipeline:", pipeline.run_query.__name__)
print("retrieve:", getattr(retrieve_similar, "__name__", type(retrieve_similar)))
print("icrl import error:", ICRL_IMPORT_ERROR)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 50)
pd.set_option("display.width", 160)

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "font.size": 11,
    }
)

print("repo:", REPO_ROOT)
print("django configured:", dj_settings.configured)
print("router base:", dj_settings.ROUTER_BASE_URL)
print("router key set:", bool(dj_settings.ROUTER_API_KEY))


icrl marshal import failed; using fewshot_skeleton.retrieve_for_prompt + tables.json graph fallback
  AttributeError: module 'logging' has no attribute 'build_traversals'
pipeline: run_query
retrieve: retrieve_for_prompt
icrl import error: module 'logging' has no attribute 'build_traversals'
repo: C:\Users\Farzaneh\Downloads\aras-ai-fewshot-text2sql\aras-ai-fewshot-text2sql
django configured: True
router base: http://127.0.0.1:8101
router key set: False


## 2. Configuration & dataset paths

Central knobs for this session. Paths are relative to the repo root. Absolute `/var/www/aras-ai-data/...` values from `configs/repro.yaml` are remapped onto `data/spider/` or `data/bird/` when those local files exist.

`LIMIT = 20` is the debug default. Set it to `None` (or `0`) for the full split. Leave `RUN_INFERENCE = False` to inspect saved evaluation files.


In [2]:
DATASET = "spider"  # "spider" or "bird"
LIMIT: int | None = 20  # None or 0 = full split
RUN_INFERENCE = False  # True only when the router key and sqlite DBs are available
SAMPLE_DB_ID = "concert_singer"
SAMPLE_QUESTION = "How many singers do we have?"

POOL_PATH = REPO_ROOT / "benchmark" / "fewshot_pool_clean_en.json"
OUTPUT_DIR = REPO_ROOT / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EVAL_JSONL = OUTPUT_DIR / "notebook_eval.jsonl"
MISMATCH_JSON = OUTPUT_DIR / "mismatch_cases.json"

LOCAL_DATA = {
    "spider": {
        "root": REPO_ROOT / "data" / "spider",
        "dev": REPO_ROOT / "data" / "spider" / "spider_data" / "dev.json",
        "train": REPO_ROOT / "data" / "spider" / "spider_data" / "train_spider.json",
        "tables": REPO_ROOT / "data" / "spider" / "spider_data" / "tables.json",
        "sqlite_dir": REPO_ROOT / "data" / "spider" / "spider_data" / "database",
    },
    "bird": {
        "root": REPO_ROOT / "data" / "bird",
        "dev": REPO_ROOT / "data" / "bird" / "dev.json",
        "train": REPO_ROOT / "data" / "bird" / "train.json",
        "tables": REPO_ROOT / "data" / "bird" / "dev_tables.json",
        "sqlite_dir": REPO_ROOT / "data" / "bird" / "dev_databases",
    },
}


def _as_path(value) -> Path:
    return Path(str(value or "")).expanduser()


def remap_legacy_path(raw: str, kind: str) -> Path:
    """Prefer the local data/ tree over the lab absolute paths in repro.yaml."""
    local = LOCAL_DATA[DATASET].get(kind)
    if local and Path(local).exists():
        return Path(local)
    text = str(raw or "").replace("\\", "/")
    marker = "data/spider/"
    if marker in text:
        suffix = text.split(marker, 1)[1]
        candidate = REPO_ROOT / "data" / "spider" / suffix
        if candidate.exists():
            return candidate
    marker = "data/bird/"
    if marker in text:
        suffix = text.split(marker, 1)[1]
        candidate = REPO_ROOT / "data" / "bird" / suffix
        if candidate.exists():
            return candidate
    path = _as_path(raw)
    return path if path.exists() else (local or path)


CFG = load_repro_config(refresh=True)
paths = CFG["paths"]
DEV_PATH = remap_legacy_path(paths.get("dev") or paths.get("items"), "dev")
TRAIN_PATH = remap_legacy_path(paths.get("train"), "train")
TABLES_PATH = remap_legacy_path(paths.get("tables"), "tables")
SQLITE_DIR = remap_legacy_path(paths.get("spider_db_dir"), "sqlite_dir")
MODEL_NAME = str(CFG.get("chat_model") or CFG.get("model") or "")
SELECTION_STRATEGY = str(CFG.get("selection_strategy") or "full")
TOP_K = int(CFG.get("k") or 5)

if LIMIT is not None and int(LIMIT) > 0:
    os.environ["SPIDER_EVAL_LIMIT"] = str(int(LIMIT))
os.environ["SPIDER_DB_DIR"] = str(SQLITE_DIR)
os.environ.setdefault("SPIDER_EVAL_RESULTS", str(OUTPUT_DIR))

print(
    pd.Series(
        {
            "dataset": DATASET,
            "limit": LIMIT,
            "run_inference": RUN_INFERENCE,
            "model": MODEL_NAME,
            "strategy": SELECTION_STRATEGY,
            "k": TOP_K,
            "dev": DEV_PATH,
            "tables": TABLES_PATH,
            "sqlite_dir": SQLITE_DIR,
            "pool": POOL_PATH,
            "output_dir": OUTPUT_DIR,
            "eval_jsonl": EVAL_JSONL,
            "mismatch_json": MISMATCH_JSON,
            "dev_exists": DEV_PATH.is_file(),
            "tables_exists": TABLES_PATH.is_file(),
            "sqlite_dir_exists": SQLITE_DIR.is_dir(),
        },
        name="config",
    ).to_frame()
)


                                                                                                                               config
dataset                                                                                                                        spider
limit                                                                                                                              20
run_inference                                                                                                                   False
model                                                                                                              gapgpt-gpt-4o-mini
strategy                                                                                                                         full
k                                                                                                                                   5
dev                    C:\Users\Farzaneh\Downloads\aras-ai-few

## 3. Schema graph & few-shot pool

Load one database (from `tables.json` when present, otherwise a tiny concert_singer fallback), run `build_traversals`, then show generated schema paths and retrieved few-shot examples as DataFrames.


In [3]:
FALLBACK_TABLES = {
    "db_id": "concert_singer",
    "table_names_original": ["stadium", "singer", "concert", "singer_in_concert"],
    "column_names_original": [
        [-1, "*"],
        [0, "Stadium_ID"],
        [0, "Location"],
        [0, "Name"],
        [0, "Capacity"],
        [0, "Highest"],
        [0, "Lowest"],
        [0, "Average"],
        [1, "Singer_ID"],
        [1, "Name"],
        [1, "Country"],
        [1, "Song_Name"],
        [1, "Song_release_year"],
        [1, "Age"],
        [1, "Is_male"],
        [2, "concert_ID"],
        [2, "concert_Name"],
        [2, "Theme"],
        [2, "Stadium_ID"],
        [2, "Year"],
        [3, "concert_ID"],
        [3, "Singer_ID"],
    ],
    "foreign_keys": [[18, 1], [20, 15], [21, 8]],
}


def load_json_records(path: Path):
    if not path or not Path(path).is_file():
        return []
    text = Path(path).read_text(encoding="utf-8").strip()
    if not text:
        return []
    if path.suffix.lower() == ".jsonl" or text.startswith("{") and "\n{" in text[:2000]:
        rows = []
        for line in text.splitlines():
            line = line.strip()
            if line:
                rows.append(json.loads(line))
        return rows
    payload = json.loads(text)
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        for key in ("results", "rows", "items", "data", "mismatch_cases", "cases"):
            if isinstance(payload.get(key), list):
                return payload[key]
        return [payload]
    return []


def spider_tables_entry(db_id: str) -> dict:
    if TABLES_PATH.is_file():
        tables = json.loads(TABLES_PATH.read_text(encoding="utf-8"))
        for entry in tables:
            if str(entry.get("db_id") or "") == db_id:
                return entry
        if tables:
            return tables[0]
    return dict(FALLBACK_TABLES)


def catalog_from_spider(entry: dict) -> tuple[dict, list[dict], dict[str, list[str]]]:
    names = list(entry.get("table_names_original") or entry.get("table_names") or [])
    catalog: dict[str, list[dict]] = {name: [] for name in names}
    schema_map: dict[str, list[str]] = {name: [] for name in names}
    for table_idx, col_name in entry.get("column_names_original") or entry.get("column_names") or []:
        if int(table_idx) < 0:
            continue
        table = names[int(table_idx)]
        rec = {
            "name": col_name,
            "english_name": col_name,
            "persian_name": col_name,
            "sql_type": "",
            "unit": "",
            "format": "",
        }
        catalog[table].append(rec)
        schema_map[table].append(str(col_name).lower())
    tables_meta = []
    for table in names:
        tables_meta.append(
            {
                "table": table,
                "columns": [c["name"] for c in catalog[table]],
                "foreign_keys": [],
            }
        )
    col_lookup = []
    for table_idx, col_name in entry.get("column_names_original") or []:
        col_lookup.append((int(table_idx), col_name))
    for src_i, dst_i in entry.get("foreign_keys") or []:
        if src_i >= len(col_lookup) or dst_i >= len(col_lookup):
            continue
        src_t, src_c = col_lookup[src_i]
        dst_t, dst_c = col_lookup[dst_i]
        if src_t < 0 or dst_t < 0:
            continue
        tables_meta[src_t].setdefault("foreign_keys", []).append(
            {
                "column": src_c,
                "references_table": names[dst_t],
                "references_column": dst_c,
            }
        )
    return catalog, tables_meta, schema_map


def schema_description(catalog: dict) -> str:
    lines = ["Linked database tables:"]
    for table, cols in catalog.items():
        labels = [c.get("english_name") or c.get("name") or "" for c in cols]
        lines.append(f"- {table}: {', '.join(labels)}")
    return "\n".join(lines)


def sqlite_path_for(db_id: str) -> Path | None:
    if not db_id:
        return None
    path = SQLITE_DIR / db_id / f"{db_id}.sqlite"
    return path if path.is_file() else None


_orig_source_to_db_config = connections_mod.source_to_db_config


def source_to_db_config_notebook(source: DataSource) -> dict:
    path = getattr(source, "_nb_sqlite_path", None)
    if path:
        return {
            "dialect": "mysql",
            "host": "localhost",
            "port": 0,
            "user": "sqlite",
            "password": "",
            "database": str(path),
            "_sqlite_path": str(path),
        }
    return _orig_source_to_db_config(source)


connections_mod.source_to_db_config = source_to_db_config_notebook
pipeline.source_to_db_config = source_to_db_config_notebook


def make_source(db_id: str) -> DataSource:
    entry = spider_tables_entry(db_id)
    resolved_id = str(entry.get("db_id") or db_id)
    catalog, tables_meta, _schema_map = catalog_from_spider(entry)
    source = DataSource(
        tenant="notebook",
        owner_external_id="local",
        type=DataSourceType.UPLOAD,
        title=resolved_id,
        description=f"Notebook source for {resolved_id}",
        schema_description=schema_description(catalog),
        db_engine="mysql",
        db_name=resolved_id,
        meta={"tables": tables_meta, "db_id": resolved_id},
        column_catalog=catalog,
    )
    sqlite = sqlite_path_for(resolved_id)
    if sqlite is not None:
        source._nb_sqlite_path = str(sqlite)
    return source


def pool_examples(path: Path = POOL_PATH, *, limit: int | None = None) -> list[SimpleNamespace]:
    rows = load_json_records(path)
    examples = []
    for i, row in enumerate(rows, start=1):
        if not isinstance(row, dict):
            continue
        question = str(row.get("user_question") or row.get("question") or "").strip()
        sql = str(row.get("ideal_sql") or row.get("query") or row.get("gold_sql") or "").strip()
        if not question or not sql:
            continue
        examples.append(
            SimpleNamespace(
                id=int(row.get("id") or i),
                user_question=question,
                ideal_sql=sql,
                ideal_answer=str(row.get("ideal_answer") or ""),
                intent=str(row.get("intent") or ""),
                priority=int(row.get("priority") or 100),
                is_active=bool(row.get("is_active", True)),
                db_id=str(row.get("db_id") or ""),
                official_hardness=row.get("official_hardness") or row.get("hardness"),
            )
        )
        if limit is not None and len(examples) >= int(limit):
            break
    return examples


def examples_dataframe(examples: list) -> pd.DataFrame:
    records = []
    for ex in examples:
        skel = sql_skeleton(getattr(ex, "ideal_sql", "") or "")
        records.append(
            {
                "id": getattr(ex, "id", None),
                "db_id": getattr(ex, "db_id", "") or "",
                "question": getattr(ex, "user_question", ""),
                "ideal_sql": getattr(ex, "ideal_sql", ""),
                "intent": getattr(ex, "intent", ""),
                "priority": getattr(ex, "priority", None),
                "band": getattr(skel, "band", None) if skel is not None else None,
                "join_count": getattr(skel, "join_count", None) if skel is not None else None,
                "tables": ", ".join(sorted(getattr(skel, "tables", []) or [])) if skel else "",
            }
        )
    return pd.DataFrame.from_records(records)


def patch_fewshots(examples: list) -> None:
    """Keep ORM off the notebook path; inject the in-memory pool into pipeline retrieval."""

    def _list(_source):
        return list(examples)

    def _match(_source, question: str):
        q = (question or "").strip()
        for ex in examples:
            if (getattr(ex, "user_question", "") or "").strip() == q:
                return ex
        return None

    pipeline._list_active_fewshots = _list  # type: ignore[method-assign]
    pipeline._fewshot_match = _match  # type: ignore[method-assign]


source = make_source(SAMPLE_DB_ID)
schema_map_by_db = {source.title: catalog_from_spider(spider_tables_entry(source.title))[2]}
max_trav = int(CFG.get("icrl", {}).get("max_traversals") or 8)
traversals = build_traversals(source.meta, max_traversals=max_trav)
table_count, pair_count = schema_join_stats(source.meta)

trav_rows = []
for trav in traversals:
    joins = [
        f"{src}.{src_col} = {dst}.{dst_col}"
        for src, src_col, dst, dst_col in (getattr(trav, "foreign_keys", None) or [])
    ]
    trav_rows.append(
        {
            "tables": ", ".join(trav.tables),
            "n_tables": len(trav.tables),
            "n_joins": len(joins),
            "joins": " | ".join(joins),
            "snippet": trav.schema_snippet(),
        }
    )
traversal_df = pd.DataFrame(trav_rows)
print(f"db_id={source.title}  tables={table_count}  connected_pairs={pair_count}  traversals={len(traversals)}")
display(traversal_df.head(20) if not traversal_df.empty else pd.DataFrame({"note": ["no traversals"]}))

all_examples = pool_examples(POOL_PATH)
patch_fewshots(all_examples)
print(f"few-shot pool: {len(all_examples)} examples from {POOL_PATH.name}")
if all_examples:
    retrieved = retrieve_similar(
        SAMPLE_QUESTION,
        all_examples,
        top_k=TOP_K,
        catalog=source.column_catalog,
    )
    print(f"retrieved for {SAMPLE_QUESTION!r}: {len(retrieved)}")
    display(examples_dataframe(retrieved if retrieved else all_examples[:10]))
else:
    display(pd.DataFrame({"note": [f"pool not found or empty: {POOL_PATH}"]}))


db_id=concert_singer  tables=4  connected_pairs=3  traversals=7


,tables,n_tables,n_joins,joins,snippet
0,stadium,1,0,,"- stadium: Stadium_ID, Location, Name, Capacity, Highest, Lowest, Average"
1,singer,1,0,,"- singer: Singer_ID, Name, Country, Song_Name, Song_release_year, Age, Is_male"
2,concert,1,0,,"- concert: concert_ID, concert_Name, Theme, Stadium_ID, Year"
3,singer_in_concert,1,0,,"- singer_in_concert: concert_ID, Singer_ID"
4,"concert, stadium",2,1,concert.Stadium_ID = stadium.Stadium_ID,"- concert: concert_ID, concert_Name, Theme, Stadium_ID, Year\n- stadium: Stadium_ID, Location, Name, Capacity, Highest, Lowest, Average\nJoins:\n- concert.Stadium_ID = stadium.Stadium_ID"
5,"singer_in_concert, concert",2,1,singer_in_concert.concert_ID = concert.concert_ID,"- singer_in_concert: concert_ID, Singer_ID\n- concert: concert_ID, concert_Name, Theme, Stadium_ID, Year\nJoins:\n- singer_in_concert.concert_ID = concert.concert_ID"
6,"singer_in_concert, singer",2,1,singer_in_concert.Singer_ID = singer.Singer_ID,"- singer_in_concert: concert_ID, Singer_ID\n- singer: Singer_ID, Name, Country, Song_Name, Song_release_year, Age, Is_male\nJoins:\n- singer_in_concert.Singer_ID = singer.Singer_ID"


few-shot pool: 669 examples from fewshot_pool_clean_en.json
retrieved for 'How many singers do we have?': 1


,id,db_id,question,ideal_sql,intent,priority,band,join_count,tables
0,240,,How many active coefficients do we have?,SELECT COUNT(*) AS active_factors FROM organization_factor WHERE is_active = 1,count,1,easy,0,organization_factor


## 4. Single-query inference & debugging

Run one question through `pipeline.run_query`. The cell prints the SQL generation prompt, predicted SQL, execution preview, retry trace, and gold SQL when the sample is in the local split.

Skipped automatically when `RUN_INFERENCE` is false or the router key is missing.


In [4]:
def load_split(path: Path) -> list[dict]:
    rows = load_json_records(path)
    return [row for row in rows if isinstance(row, dict)]


def find_gold(question: str, db_id: str | None = None) -> dict | None:
    if not DEV_PATH.is_file():
        return None
    q = (question or "").strip()
    for item in load_split(DEV_PATH):
        if str(item.get("question") or "").strip() != q:
            continue
        if db_id and str(item.get("db_id") or "") not in {"", str(db_id)}:
            continue
        return item
    return None


def capture_sql_prompt() -> list[str]:
    prompts: list[str] = []
    original = pipeline._sql_generation_prompt

    def wrapped(**kwargs):
        text = original(**kwargs)
        prompts.append(text)
        return text

    pipeline._sql_generation_prompt = wrapped  # type: ignore[method-assign]
    return prompts, original


def pred_sql_from_result(result: dict) -> str:
    queries = result.get("queries") or []
    for query in reversed(queries):
        if query.get("status") == "executed" and query.get("sql"):
            return str(query["sql"])
    if queries and queries[0].get("sql"):
        return str(queries[0]["sql"])
    return ""


def print_retry_trace(result: dict) -> None:
    meta = result.get("metadata") or {}
    print("sql_retry_attempted:", meta.get("sql_retry_attempted"))
    print("sql_retry_fixed:", meta.get("sql_retry_fixed"))
    print("fewshot_ids:", meta.get("fewshot_ids"))
    print("fewshot_draft_band:", meta.get("fewshot_draft_band"))
    queries = result.get("queries") or []
    if not queries:
        print("queries: (none)")
        return
    for query in queries:
        print(
            f"- attempt={query.get('attempt')} status={query.get('status')} "
            f"repaired={query.get('repaired')} error={query.get('error') or ''}"
        )
        if query.get("sql"):
            print(query["sql"])


single_result = None
gold_item = find_gold(SAMPLE_QUESTION, SAMPLE_DB_ID)
gold_sql = str((gold_item or {}).get("query") or (gold_item or {}).get("gold_sql") or "")

if not RUN_INFERENCE:
    print("RUN_INFERENCE is False — skip live router call. Load saved eval files in section 5.")
elif not dj_settings.ROUTER_API_KEY:
    print("No ROUTER_API_KEY / OPENAI_API_KEY in the environment — skip live inference.")
else:
    captured, original_prompt_fn = capture_sql_prompt()
    stages: list[str] = []

    def progress(stage, status, **kwargs):
        stages.append(f"{stage}:{status}")

    try:
        single_result = pipeline.run_query(
            source=source,
            question=SAMPLE_QUESTION,
            options={
                "model": MODEL_NAME,
                "include_chart": False,
                "skip_planner": False,
            },
            progress=progress,
        )
    finally:
        pipeline._sql_generation_prompt = original_prompt_fn  # type: ignore[method-assign]

    pred_sql = pred_sql_from_result(single_result or {})
    print("=== stages ===")
    print(" -> ".join(stages) or "(none)")
    print("\n=== SQL generation prompt ===")
    print(captured[-1] if captured else "(prompt not captured; exact-match few-shot may have been used)")
    print("\n=== predicted SQL ===")
    print(pred_sql or "(none)")
    print("\n=== gold SQL ===")
    print(gold_sql or "(not in local split / split file missing)")
    print("\n=== retry trace ===")
    print_retry_trace(single_result or {})
    print("\n=== execution preview ===")
    data = (single_result or {}).get("data") or {}
    preview_df = pd.DataFrame(data.get("rows") or [], columns=data.get("columns") or None)
    display(preview_df.head(20) if not preview_df.empty else pd.DataFrame({"note": ["no rows"]}))
    print("\n=== grounded answer ===")
    print((single_result or {}).get("answer") or "")
    if gold_sql and pred_sql:
        print("\n=== gold vs pred (normalized whitespace) ===")
        same = " ".join(gold_sql.lower().split()) == " ".join(pred_sql.lower().split())
        print("whitespace-normalized equal:", same)


RUN_INFERENCE is False — skip live router call. Load saved eval files in section 5.


## 5. Batch benchmark evaluation

If `RUN_INFERENCE` is true, run the evaluation loop on `dev.json` (honouring `LIMIT`) and write `results/notebook_eval.jsonl` plus `results/mismatch_cases.json`.

Otherwise load an existing evaluation JSON / JSONL or `mismatch_cases.json` so the tables and charts still run.


In [ ]:
def first_existing(*candidates: Path) -> Path | None:
    for path in candidates:
        if path and Path(path).is_file() and Path(path).stat().st_size > 0:
            return Path(path)
    return None


def discover_saved_eval() -> list[dict]:
    found = first_existing(
        EVAL_JSONL,
        OUTPUT_DIR / "eval.jsonl",
        OUTPUT_DIR / "eval.json",
        MISMATCH_JSON,
        REPO_ROOT / "benchmark" / "eval_spider_service_new.jsonl",
    )
    if found is None:
        jsonl = sorted(OUTPUT_DIR.glob("*.jsonl"), key=lambda p: p.stat().st_mtime, reverse=True)
        json_files = sorted(OUTPUT_DIR.glob("*.json"), key=lambda p: p.stat().st_mtime, reverse=True)
        found = (jsonl + json_files)[0] if (jsonl or json_files) else None
    if found is None:
        print("No saved evaluation file under results/. Charts stay empty until you run inference or copy a jsonl here.")
        return []
    print(f"loaded evaluation file: {found}")
    return load_json_records(found)


def result_sets_equal(a: list, b: list) -> bool:
    def norm(rows):
        return sorted(tuple("" if v is None else str(v) for v in row) for row in rows)

    try:
        return norm(a) == norm(b)
    except Exception:
        return False


def homemade_ex(db_id: str, gold_sql: str, pred_sql: str) -> int | None:
    sqlite = sqlite_path_for(db_id)
    if sqlite is None or not gold_sql or not pred_sql:
        return None
    db_conf = {
        "dialect": "mysql",
        "host": "localhost",
        "port": 0,
        "user": "sqlite",
        "password": "",
        "database": str(sqlite),
        "_sqlite_path": str(sqlite),
    }
    try:
        _gc, gold_rows, _ = connections_mod.execute_readonly_sql(db_conf, gold_sql, timeout=30)
        _pc, pred_rows, _ = connections_mod.execute_readonly_sql(db_conf, pred_sql, timeout=30)
    except Exception:
        return 0
    return int(result_sets_equal(gold_rows, pred_rows))


def schema_map_for(db_id: str) -> dict[str, list[str]]:
    if db_id in schema_map_by_db:
        return schema_map_by_db[db_id]
    entry = spider_tables_entry(db_id)
    _catalog, _meta, schema_map = catalog_from_spider(entry)
    schema_map_by_db[str(entry.get("db_id") or db_id)] = schema_map
    return schema_map


def official_scores(db_id: str, gold_sql: str, pred_sql: str) -> dict:
    try:
        return score_sql(gold_sql, pred_sql, schema_map_for(db_id), db_id=db_id)
    except Exception as exc:
        return {"exact_match": None, "official_ex": None, "official_error": str(exc)[:240]}


def difficulty_of(row: dict, gold_sql: str) -> str | None:
    for key in ("official_hardness", "difficulty", "hardness", "level"):
        value = row.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip().lower()
    skel = sql_skeleton(gold_sql or "")
    return getattr(skel, "band", None) if skel is not None else None


def error_type_of(row: dict, result: dict | None = None) -> str:
    reason = row.get("error_type") or row.get("reason") or row.get("status")
    if reason:
        return str(reason)
    queries = (result or {}).get("queries") or row.get("queries") or []
    if queries:
        last = queries[-1] if isinstance(queries[-1], dict) else {}
        if last.get("status") in {"blocked", "error"}:
            return str(last.get("status"))
        if last.get("error"):
            return "error"
    if row.get("ex") in (0,) or row.get("score") in (0,):
        return "result_mismatch"
    if not (row.get("pred_sql") or row.get("predicted_sql")):
        return "no_sql"
    return ""


def eval_one(item: dict) -> dict:
    db_id = str(item.get("db_id") or SAMPLE_DB_ID)
    question = str(item.get("question") or "").strip()
    gold_sql = str(item.get("query") or item.get("gold_sql") or "")
    src = make_source(db_id)
    result = pipeline.run_query(
        source=src,
        question=question,
        options={"model": MODEL_NAME, "include_chart": False, "skip_planner": False},
    )
    pred_sql = pred_sql_from_result(result)
    meta = result.get("metadata") or {}
    official = official_scores(db_id, gold_sql, pred_sql)
    ex = official.get("official_ex")
    if ex not in (0, 1):
        ex = homemade_ex(db_id, gold_sql, pred_sql)
    record = {
        "db_id": db_id,
        "question": question,
        "gold_sql": gold_sql,
        "pred_sql": pred_sql,
        "predicted_sql": pred_sql,
        "answer": result.get("answer") or "",
        "ex": ex,
        "score": ex,
        "exact_match": official.get("exact_match"),
        "official_ex": official.get("official_ex"),
        "official_error": official.get("official_error"),
        "difficulty": difficulty_of(item, gold_sql),
        "official_hardness": item.get("sql") and None,
        "sql_retry_attempted": bool(meta.get("sql_retry_attempted")),
        "sql_retry_fixed": bool(meta.get("sql_retry_fixed")),
        "fewshot_ids": meta.get("fewshot_ids") or [],
        "fewshot_retrieved": meta.get("fewshot_retrieved"),
        "queries": result.get("queries") or [],
        "error": meta.get("error") or "",
    }
    record["error_type"] = error_type_of(record, result)
    if item.get("sql") and isinstance(item.get("sql"), dict):
        try:
            from benchmark.eval_instrumentation import get_hardness

            record["official_hardness"] = get_hardness(item["sql"])
            record["difficulty"] = record["official_hardness"]
        except Exception:
            record["difficulty"] = difficulty_of(record, gold_sql)
    else:
        record["difficulty"] = difficulty_of(record, gold_sql)
    return record


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")


def write_mismatches(path: Path, rows: list[dict]) -> None:
    mismatches = [
        row
        for row in rows
        if row.get("ex") in (0, None, False)
        or row.get("score") in (0, None, False)
        or str(row.get("error_type") or "") in {"result_mismatch", "error", "blocked", "no_sql"}
    ]
    path.write_text(json.dumps(mismatches, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    return mismatches


eval_rows: list[dict] = []
if RUN_INFERENCE and dj_settings.ROUTER_API_KEY and DEV_PATH.is_file():
    items = load_split(DEV_PATH)
    if LIMIT:
        items = items[: int(LIMIT)]
    print(f"running inference on {len(items)} question(s) from {DEV_PATH}")
    for i, item in enumerate(items, start=1):
        try:
            record = eval_one(item)
        except Exception as exc:
            record = {
                "db_id": item.get("db_id"),
                "question": item.get("question"),
                "gold_sql": item.get("query") or item.get("gold_sql"),
                "pred_sql": "",
                "ex": 0,
                "score": 0,
                "exact_match": None,
                "error_type": "error",
                "error": str(exc)[:300],
                "difficulty": difficulty_of(item, str(item.get("query") or "")),
                "sql_retry_attempted": False,
                "sql_retry_fixed": False,
            }
        eval_rows.append(record)
        mark = "ok" if record.get("ex") == 1 else record.get("error_type") or "miss"
        print(f"[{i}/{len(items)}] {record.get('db_id')} {mark}")
    write_jsonl(EVAL_JSONL, eval_rows)
    mismatches = write_mismatches(MISMATCH_JSON, eval_rows)
    print(f"wrote {EVAL_JSONL} ({len(eval_rows)} rows)")
    print(f"wrote {MISMATCH_JSON} ({len(mismatches)} mismatches)")
    print(summarize_metrics(eval_rows))
else:
    eval_rows = discover_saved_eval()
    print(f"evaluation rows loaded: {len(eval_rows)}")


## 6. Results summary tables

Overall metrics (`EX`, `ESM`, `SQL_RETRY`) and breakdowns by difficulty and `db_id`. Missing optional fields (`difficulty`, `exact_match` / ESM) are reported as n/a rather than zeros.


In [ ]:
def _first(row: dict, *keys, default=None):
    for key in keys:
        if key in row and row[key] not in (None, ""):
            return row[key]
    return default


def normalize_eval_frame(rows: list[dict]) -> pd.DataFrame:
    records = []
    for row in rows:
        if not isinstance(row, dict):
            continue
        gold = str(_first(row, "gold_sql", "query", default="") or "")
        pred = str(_first(row, "pred_sql", "predicted_sql", default="") or "")
        ex = _first(row, "official_ex", "ex", "score")
        esm = _first(row, "exact_match", "esm", "exact_set_match")
        difficulty = _first(row, "official_hardness", "difficulty", "hardness", "level")
        if not difficulty:
            difficulty = difficulty_of(row, gold)
        records.append(
            {
                "db_id": str(_first(row, "db_id", default="") or ""),
                "question": str(_first(row, "question", default="") or ""),
                "gold_sql": gold,
                "pred_sql": pred,
                "ex": int(ex) if ex in (0, 1, True, False) else pd.NA,
                "esm": int(esm) if esm in (0, 1, True, False) else pd.NA,
                "sql_retry_attempted": bool(row.get("sql_retry_attempted")),
                "sql_retry_fixed": bool(row.get("sql_retry_fixed")),
                "error_type": error_type_of(row),
                "difficulty": (str(difficulty).lower() if difficulty else pd.NA),
                "error": str(row.get("error") or row.get("official_error") or ""),
            }
        )
    frame = pd.DataFrame.from_records(records)
    if frame.empty:
        return pd.DataFrame(
            columns=[
                "db_id",
                "question",
                "gold_sql",
                "pred_sql",
                "ex",
                "esm",
                "sql_retry_attempted",
                "sql_retry_fixed",
                "error_type",
                "difficulty",
                "error",
            ]
        )
    return frame


def metric_count(series: pd.Series) -> tuple[int | None, int | None, float | None]:
    values = series.dropna()
    if values.empty:
        return None, None, None
    numbered = pd.to_numeric(values, errors="coerce").dropna()
    if numbered.empty:
        return None, None, None
    hits = int((numbered == 1).sum())
    total = int(len(numbered))
    return hits, total, 100.0 * hits / total if total else None


def fmt_count(hits, total, pct) -> str:
    if hits is None or total is None:
        return "n/a"
    return f"{hits}/{total} ({pct:.1f}%)"


eval_df = normalize_eval_frame(eval_rows)
total_queries = int(len(eval_df))
ex_hits, ex_total, ex_pct = metric_count(eval_df["ex"])
esm_hits, esm_total, esm_pct = metric_count(eval_df["esm"])
retry_attempted = int(eval_df["sql_retry_attempted"].fillna(False).astype(bool).sum()) if total_queries else 0
retry_fixed = int(eval_df["sql_retry_fixed"].fillna(False).astype(bool).sum()) if total_queries else 0
retry_pct = (100.0 * retry_fixed / retry_attempted) if retry_attempted else None

overall_df = pd.DataFrame(
    [
        {
            "Total Queries": total_queries,
            "EX count": ex_hits if ex_hits is not None else pd.NA,
            "EX %": None if ex_pct is None else round(ex_pct, 2),
            "EX": fmt_count(ex_hits, ex_total, ex_pct or 0.0),
            "ESM count": esm_hits if esm_hits is not None else pd.NA,
            "ESM %": None if esm_pct is None else round(esm_pct, 2),
            "ESM": fmt_count(esm_hits, esm_total, esm_pct or 0.0),
            "SQL_RETRY attempted": retry_attempted,
            "SQL_RETRY fixed": retry_fixed,
            "Retry recovery %": None if retry_pct is None else round(retry_pct, 2),
        }
    ]
)
print("Overall metrics")
display(overall_df)


def breakdown_table(frame: pd.DataFrame, by: str) -> pd.DataFrame:
    if frame.empty or by not in frame.columns or frame[by].dropna().empty:
        return pd.DataFrame({"note": [f"no {by} labels in this run"]})
    rows = []
    for key, part in frame.groupby(by, dropna=False):
        hits, total, pct = metric_count(part["ex"])
        esm_h, esm_t, esm_p = metric_count(part["esm"])
        errors = int((pd.to_numeric(part["ex"], errors="coerce") != 1).sum())
        rows.append(
            {
                by: key if pd.notna(key) else "(missing)",
                "n": int(len(part)),
                "EX": fmt_count(hits, total, pct or 0.0),
                "EX %": None if pct is None else round(pct, 2),
                "ESM": fmt_count(esm_h, esm_t, esm_p or 0.0),
                "errors": errors,
                "retry_attempted": int(part["sql_retry_attempted"].fillna(False).astype(bool).sum()),
                "retry_fixed": int(part["sql_retry_fixed"].fillna(False).astype(bool).sum()),
            }
        )
    out = pd.DataFrame(rows)
    if "EX %" in out.columns:
        out = out.sort_values(["EX %", "errors"], ascending=[True, False], na_position="last")
    return out.reset_index(drop=True)


print("Breakdown by difficulty")
difficulty_df = breakdown_table(eval_df, "difficulty")
display(difficulty_df)

print("Breakdown by db_id (lowest EX / most errors first)")
db_df = breakdown_table(eval_df, "db_id")
display(db_df.head(50))


## 7. Charts

**Chart 1** — overall `EX`, `ESM`, and retry recovery rates with percentage labels.

**Chart 2** — top 10 `db_id` values by mismatch count, plus a difficulty breakdown when those labels exist.


In [ ]:
def _bar_labels(ax, bars, values):
    for bar, value in zip(bars, values):
        if value is None or (isinstance(value, float) and pd.isna(value)):
            label = "n/a"
            height = 0
        else:
            label = f"{value:.1f}%"
            height = float(value)
        ax.text(bar.get_x() + bar.get_width() / 2, height + 1.2, label, ha="center", va="bottom", fontsize=10)


chart1_labels = ["EX", "ESM", "Retry Recovery Rate"]
chart1_values = [ex_pct, esm_pct, retry_pct]

fig, ax = plt.subplots(figsize=(7.5, 4.2))
colors = ["#2563eb", "#0f766e", "#c2410c"]
bars = ax.bar(chart1_labels, [v if v is not None else 0 for v in chart1_values], color=colors, width=0.62)
_bar_labels(ax, bars, chart1_values)
ax.set_ylim(0, 110)
ax.set_ylabel("percent")
ax.set_title("Overall metrics")
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

mismatch_mask = pd.to_numeric(eval_df["ex"], errors="coerce") != 1 if not eval_df.empty else pd.Series(dtype=bool)
mismatch_df_plot = eval_df.loc[mismatch_mask].copy() if not eval_df.empty else eval_df
top_db = (
    mismatch_df_plot["db_id"].replace("", pd.NA).dropna().value_counts().head(10)
    if not mismatch_df_plot.empty
    else pd.Series(dtype=int)
)

fig, axes = plt.subplots(1, 2 if not eval_df["difficulty"].dropna().empty else 1, figsize=(12, 4.8))
if not hasattr(axes, "__len__"):
    axes = [axes]

if top_db.empty:
    axes[0].text(0.5, 0.5, "no mismatch cases to plot", ha="center", va="center")
    axes[0].set_axis_off()
else:
    axes[0].barh(list(reversed(top_db.index.tolist())), list(reversed(top_db.values.tolist())), color="#b91c1c")
    axes[0].set_xlabel("mismatch count")
    axes[0].set_title("Top 10 databases by mismatches")

if len(axes) > 1:
    diff_counts = (
        mismatch_df_plot["difficulty"].dropna().value_counts().reindex(
            ["easy", "medium", "hard", "extra"], fill_value=0
        )
    )
    present = diff_counts[diff_counts.index.isin(mismatch_df_plot["difficulty"].dropna().unique())]
    if present.empty:
        axes[1].text(0.5, 0.5, "no difficulty labels", ha="center", va="center")
        axes[1].set_axis_off()
    else:
        axes[1].barh(list(reversed(present.index.tolist())), list(reversed(present.values.tolist())), color="#7c3aed")
        axes[1].set_xlabel("mismatch count")
        axes[1].set_title("Mismatches by difficulty")

fig.tight_layout()
plt.show()


## 8. Mismatch cases inspector

Structured table of failures. Edit `FILTER_DB_ID` or `FILTER_KEYWORD` (for example `LIMIT`, `JOIN`, `NULL`) and re-run the filter cell.


In [ ]:
mismatch_source = first_existing(MISMATCH_JSON)
if mismatch_source is not None:
    raw_mismatches = load_json_records(mismatch_source)
    print(f"loaded mismatches from {mismatch_source}")
    mismatch_df = normalize_eval_frame(raw_mismatches)
else:
    mismatch_df = eval_df.loc[pd.to_numeric(eval_df["ex"], errors="coerce") != 1].copy() if not eval_df.empty else eval_df.copy()

keep_cols = [c for c in ["db_id", "question", "gold_sql", "pred_sql", "error_type", "difficulty"] if c in mismatch_df.columns]
mismatch_view = mismatch_df[keep_cols].reset_index(drop=True) if keep_cols else mismatch_df
print(f"mismatch rows: {len(mismatch_view)}")
display(mismatch_view.head(50) if not mismatch_view.empty else pd.DataFrame({"note": ["no mismatch cases"]}))


In [ ]:
FILTER_DB_ID = ""          # e.g. "concert_singer"
FILTER_KEYWORD = "JOIN"    # applied to question + gold_sql + pred_sql; try LIMIT, JOIN, NULL

filtered = mismatch_view.copy()
if FILTER_DB_ID:
    filtered = filtered[filtered["db_id"].astype(str) == FILTER_DB_ID]
if FILTER_KEYWORD:
    needle = str(FILTER_KEYWORD)
    blob = (
        filtered.get("question", pd.Series(dtype=str)).fillna("").astype(str)
        + " "
        + filtered.get("gold_sql", pd.Series(dtype=str)).fillna("").astype(str)
        + " "
        + filtered.get("pred_sql", pd.Series(dtype=str)).fillna("").astype(str)
    )
    filtered = filtered[blob.str.contains(needle, case=False, regex=False, na=False)]

print(f"filter db_id={FILTER_DB_ID!r} keyword={FILTER_KEYWORD!r} → {len(filtered)} row(s)")
display(filtered.head(50) if not filtered.empty else pd.DataFrame({"note": ["no rows match this filter"]}))
